# Medallion Architecture Practice

Bronze → Silver → Gold: land raw data as-is, clean and conform it, then build business-ready aggregates.

In [0]:
from pyspark.sql import functions as F

# A schema for the three medallion layers
spark.sql("CREATE SCHEMA IF NOT EXISTS medallion_practice")

# BRONZE — land the data exactly as it arrives: duplicates, mixed casing,
# nulls, and stray whitespace are all kept. Bronze is "raw, immutable, as-is".
raw = spark.createDataFrame(
    [
        (101, "RAVI",    "IT",    80000),
        (101, "ravi",    "IT",    80000),   # duplicate of 101
        (102, "Priya",   "it",    95000),
        (104, "sita",    "HR",    60000),
        (105, "Arjun",   None,    75000),   # missing department
        (107, "kiran",   "Sales", None),   # missing salary
        (108, " Meena ", "Sales", 90000),   # stray whitespace
    ],
    schema="emp_id LONG, name STRING, department STRING, salary LONG",
)

bronze = raw.withColumn("ingested_at", F.current_timestamp())
bronze.write.format("delta").mode("overwrite").saveAsTable("medallion_practice.bronze_employees")

print(f"Bronze row count: {spark.table('medallion_practice.bronze_employees').count()}")
display(spark.table("medallion_practice.bronze_employees"))

Bronze row count: 7


emp_id,name,department,salary,ingested_at
101,RAVI,IT,80000,2026-11-03T08:19:52.186Z
101,ravi,IT,80000,2026-11-03T08:19:52.186Z
102,Priya,it,95000,2026-11-03T08:19:52.186Z
104,sita,HR,60000,2026-11-03T08:19:52.186Z
105,Arjun,null,75000,2026-11-03T08:19:52.186Z
107,kiran,Sales,null,2026-11-03T08:19:52.186Z
108,Meena,Sales,90000,2026-11-03T08:19:52.186Z


In [0]:
# SILVER — clean and conform the data: trim and standardize casing,
# drop duplicate employees, and remove rows with missing business fields.
silver = (
    spark.table("medallion_practice.bronze_employees")
    .select(
        F.col("emp_id"),
        F.trim(F.initcap(F.col("name"))).alias("name"),
        F.initcap(F.col("department")).alias("department"),
        F.col("salary"),
    )
    .filter(F.col("salary").isNotNull() & (F.col("salary") > 0) & F.col("department").isNotNull())
    .dropDuplicates(["emp_id"])
)

silver.write.format("delta").mode("overwrite").saveAsTable("medallion_practice.silver_employees")

print(f"Silver row count: {spark.table('medallion_practice.silver_employees').count()}")
display(spark.table("medallion_practice.silver_employees"))

Silver row count: 4


emp_id,name,department,salary
102,Priya,It,95000
104,Sita,Hr,60000
101,Ravi,It,80000
108,Meena,Sales,90000


In [0]:
# GOLD — business-ready aggregates: one row per department for reporting.
gold = (
    spark.table("medallion_practice.silver_employees")
    .groupBy("department")
    .agg(
        F.count("*").alias("num_employees"),
        F.round(F.avg("salary"), 0).alias("avg_salary"),
        F.min("salary").alias("min_salary"),
        F.max("salary").alias("max_salary"),
    )
    .orderBy("department")
)

gold.write.format("delta").mode("overwrite").saveAsTable("medallion_practice.gold_dept_summary")

display(spark.table("medallion_practice.gold_dept_summary"))

department,num_employees,avg_salary,min_salary,max_salary
Hr,1,60000.0,60000,60000
It,2,87500.0,80000,95000
Sales,1,90000.0,90000,90000


In [0]:
# The three layers side by side
for layer in ["bronze_employees", "silver_employees", "gold_dept_summary"]:
    t = f"medallion_practice.{layer}"
    print(f"{t}: {spark.table(t).count()} rows")

display(spark.sql("SHOW TABLES IN medallion_practice"))

medallion_practice.bronze_employees: 7 rows
medallion_practice.silver_employees: 4 rows
medallion_practice.gold_dept_summary: 3 rows


database,tableName,isTemporary
medallion_practice,bronze_employees,false
medallion_practice,gold_dept_summary,false
medallion_practice,silver_employees,false


## CSV parsing modes — PERMISSIVE, DROPMALFORMED, FAILFAST

How Spark handles malformed records when reading CSV files: keep with nulls, silently drop, or fail the read.

In [0]:
# Write a deliberately malformed CSV using Spark itself (no header line —
# the schema is supplied explicitly at read time, so file order doesn't matter):
#   row 3 has a non-numeric salary, row 4 is missing a field entirely
csv_lines = [
    "1,Ravi,80000",
    "2,Priya,95000",
    "3,BadRow,not_a_number",
    "4,ShortRow",
]

# Explicit schema: what each row SHOULD parse to
csv_schema = "emp_id INT, name STRING, salary INT"

lines = spark.createDataFrame([(l,) for l in csv_lines], schema="line STRING")
path = "/Volumes/workspace/medallion_practice/demo_files/malformed_csv"
lines.write.mode("overwrite").text(path)

# Sanity-check what landed on disk
spark.read.text(path).show(truncate=False)

+---------------------+
|value                |
+---------------------+
|3,BadRow,not_a_number|
|2,Priya,95000        |
|1,Ravi,80000         |
|4,ShortRow           |
+---------------------+



In [0]:
# PERMISSIVE (the default) — keep every row; values that can't be parsed
# become null. For CSV, _corrupt_record must be part of the schema to capture
# the raw offending line.
permissive_schema = csv_schema + ", _corrupt_record STRING"
permissive = (
    spark.read
    .schema(permissive_schema)
    .option("mode", "PERMISSIVE")
    .csv(path)
)
permissive.show(truncate=False)
print(f"Rows returned: {permissive.count()}")

+------+--------+------+---------------------+
|emp_id|name    |salary|_corrupt_record      |
+------+--------+------+---------------------+
|3     |BadRow  |NULL  |3,BadRow,not_a_number|
|2     |Priya   |95000 |NULL                 |
|1     |Ravi    |80000 |NULL                 |
|4     |ShortRow|NULL  |4,ShortRow           |
+------+--------+------+---------------------+

Rows returned: 4


In [0]:
# DROPMALFORMED — rows that can't be parsed to the schema are silently dropped
dropmalformed = (
    spark.read
    .schema(csv_schema)
    .option("mode", "DROPMALFORMED")
    .csv(path)
)
# Note: .count() on a CSV file source can be line-count-optimized — it counts
# raw lines WITHOUT parsing, so it returns 4 here even though 2 rows were
# dropped. Only row-producing actions (show/display) reflect the drop.
display(dropmalformed)

emp_id,name,salary
2,Priya,95000
1,Ravi,80000


In [0]:
# FAILFAST — abort the read as soon as a malformed record is found.
# The error surfaces at action time (show/count), not at read time.
failfast_df = (
    spark.read
    .schema(csv_schema)
    .option("mode", "FAILFAST")
    .csv(path)
)
try:
    failfast_df.show(truncate=False)
except Exception as e:
    print("FAILFAST raised (expected):")
    print(str(e).split("\n")[0])

FAILFAST raised (expected):
[FAILED_READ_FILE.NO_HINT] Error while reading file dbfs:/Volumes/workspace/medallion_practice/demo_files/malformed_csv/part-00002-tid-921315593679346376-35ee18de-8dbd-4dd1-8dae-055991307f29-201-1-c000.txt.  SQLSTATE: KD001
